# ASR MODEL COMPARISON


In [6]:
!pip install faster-whisper jiwer rapidfuzz requests pandas plotly unidecode

In [14]:
import os
import getpass
from google.colab import files

DEEPGRAM_API_KEY = "dfb5c22691249c1f61832bb8364f710ebc82568e"
SARVAM_API_KEY = "sk_je4t3vpn_GHQZR2NBWkRFnI5meJGzKa8K"

AUDIO_DIR = "/content/audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

In [15]:
import time, re, unicodedata, json, math
from pathlib import Path
from collections import Counter
import pandas as pd
import requests
from rapidfuzz import fuzz
from faster_whisper import WhisperModel
from unidecode import unidecode

OUTPUT_CSV  = "benchmark_results.csv"
OUTPUT_HTML = "benchmark_report.html"

NOISE_CONDITIONS = {
    'Bellandur' : "quiet",
    'Btm layout' : "quiet",
    'Chruch street': "noisy",
    'Electronic city': "noisy",
    'Electronic city2': "quite",
    'Hsr': "nosiy",
    'Jp nagar': "quiet",
    'Kengeri': "quiet",
    'Koramangala': "quiet",
    'MTR hotel': "nosiy",
    'Majestic,silk board, koramangala' : "quiet",
    'Majestic': "quiet",
    'Marathahalli':  "noisy",
    'Mg road , indiranagar': "nosiy",
    'Peenya': "quiet",
    'Rr nagar': "quiet",
    'Whitefeild': "quiet",
    'Whitefield, marathahalli': 'nosiy',
    'Yalahanka': 'quiet',
    'call': 'nosiy'
}
DEFAULT_NOISE = "quiet"

MANUAL_LOCALITIES ={
    'Bellandur' : ["Bellandur"],
    'Btm layout' : ["BTM Layout"],
    'Chruch street': ["Chruch Street"],
    'Electronic city': ["Electronic city"],
    'Electronic city2': ["Electronic city"],
    'Hsr': ["HSR Layout"],
    'Jp nagar': ["JP Nagar"],
    'Kengeri': ["Kengeri"],
    'Koramangala': ["Koramangala"],
    'MTR hotel': ["MTR Hotel"],
    'Majestic,silk board, koramangala' : ["Majestic","Silk Board", "Koramangala"],
    'Majestic': ["Majestic"],
    'Marathahalli': ["Marathahalli"],
    'Mg road , indiranagar': ["MG road", "Indiranagar"],
    'Peenya': ["Peenya"],
    'Rr nagar': ["RR Nagar"],
    'Whitefeild': ["Whitefeild"],
    'Whitefield, marathahalli': ["Whitefield", "Marathahalli"],
    'Yalahanka': ["Yalahanka"],
    'call': []
}

LOCALITIES = {
    "Koramangala":          ["koramangala","koromangala","koramangla", "कोरमंगला", "कोरामंगला"],
    "Indiranagar":          ["indiranagar","indira nagar","indranagar", "इंदिरानगर", "इंद्रा नगर"],
    "Whitefield":           ["whitefield","white field","whitefeild", "वाइटफील्ड", "व्हाइटफील्ड"],
    "Electronic City":      ["electronic city","electronics city","e city", "इलेक्ट्रॉनिक सिटी", "ई सिटी"],
    "Marathahalli":         ["marathahalli","marathalli","maradahalli", "मराठल्ली", "मारथहल्ली"],
    "Jayanagar":            ["jayanagar","jaya nagar", "जयनगर", "जया नगर"],
    "Rajajinagar":          ["rajajinagar","raja ji nagar", "राजाजीनगर", "राजाजी नगर"],
    "Hebbal":               ["hebbal","hebal","hebball", "हेब्बल", "हेब्बाल"],
    "Yelahanka":            ["yelahanka","yelahanca", "येलहंका", "येलाहंका"],
    "Banashankari":         ["banashankari","bana shankari", "बनशंकरी", "बनाशंकरी"],
    "HSR Layout":           ["hsr layout","h s r layout","hsr", "एचएसआर लेआउट", "एच एस आर"],
    "BTM Layout":           ["btm layout","b t m layout","btm", "बीटीएम लेआउट", "बी टी एम"],
    "Majestic":             ["majestic","majestik", "मैजेस्टिक", "मजेस्टिक"],
    "Silk Board":           ["silk board","silkboard", "सिल्क बोर्ड", "सिल्कबोर्ड"],
    "Bellandur":            ["bellandur","bellndur", "बेलंदूर", "बेलन्दुर"],
    "Sarjapur":             ["sarjapur","sarjapura", "सरजापुर", "सर्जापुर"],
    "Bommanahalli":         ["bommanahalli","bommana halli", "बोम्मनहल्ली", "बोम्मन हल्ली"],
    "KR Puram":             ["kr puram","k r puram", "केआर पुरम", "के आर पुरम"],
    "Peenya":               ["peenya","penya", "पीन्या", "पीनिया"],
    "Yeshwanthpur":         ["yeshwanthpur","yeshwantpur", "यशवंतपुर", "यश्वंतपुर"],
    "Byatarayanapura":      ["byatarayanapura","byatarayana pura", "ब्याटरायनपुरा", "ब्याटरनपुरा"],
    "Kadugondanahalli":     ["kadugondanahalli","kadugondana halli", "काडुगोंडनहल्ली", "काडुगोंडना हल्ली"],
    "Hesaraghatta":         ["hesaraghatta","hesaragatta", "हेसरघट्टा", "हेसरघटा"],
    "Chikkabanavara":       ["chikkabanavara","chikka banavara", "चिक्कबाणावर", "चिक्कबानावरा"],
    "Rajarajeshwarinagar":  ["rajarajeshwarinagar","raja rajeshwari nagar", "राजराजेश्वरीनगर", "राज राजेश्वरी नगर"],
    "Kothanur Dinne":       ["kothanur dinne","kothanur", "कोथनूर डिन्ने", "कोथनूर"],
    "Thanisandra":          ["thanisandra","thanisandara", "थानिसंड्रा", "तनिसंद्रा"],
    "Doddanekundi":         ["doddanekundi","dodda nekundi", "डोड्डनेकुंडी", "दोड्डनेकुंडी"],
    "Kengeri Upanagara":    ["kengeri upanagara","kengeri", "केंगेरी उपनगर", "केंगेरी"],
    "Thalaghattapura":      ["thalaghattapura","thalaghatta pura", "तलगट्टापुरा", "थालाघट्टापुरा"],
    "MTR Hotel":            ["mtr","M T R", "एमटीआर होटल"]
}

ALL_VARIANTS = {}
for canonical, variants in LOCALITIES.items():
    for v in variants:
        ALL_VARIANTS[v] = canonical

def normalize(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[.,!?\"'|।\-\(\)\[\]{}:;]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def detect_localities(transcript: str) -> list[str]:
    t = normalize(transcript)
    found = set()
    for variant, canonical in ALL_VARIANTS.items():
        if variant in t:
            found.add(canonical)
    tokens = t.split()
    for window_size in [1, 2, 3]:
        for i in range(len(tokens) - window_size + 1):
            window = " ".join(tokens[i:i+window_size])
            for variant, canonical in ALL_VARIANTS.items():
                if canonical not in found:
                    score = fuzz.ratio(window, variant)
                    if score >= 80:
                        found.add(canonical)
    return sorted(found)

def cross_model_agreement(texts: list[str]) -> float:
    if len(texts) < 2: return 0.0
    pairs = []
    for i in range(len(texts)):
        for j in range(i+1, len(texts)):
            a, b = normalize(texts[i]), normalize(texts[j])
            if a and b:
                pairs.append(fuzz.token_sort_ratio(a, b))
    return round(sum(pairs) / len(pairs), 1) if pairs else 0.0

def transcript_entropy(texts: list[str]) -> float:
    all_words = []
    for t in texts:
        all_words.extend(normalize(t).split())
    if not all_words: return 0.0
    counts = Counter(all_words)
    total = len(all_words)
    entropy = -sum((c/total) * math.log2(c/total) for c in counts.values() if c > 0)
    max_entropy = math.log2(total) if total > 1 else 1
    return round(entropy / max_entropy, 3) if max_entropy > 0 else 0.0

def phoneme_stability_score(texts: list[str]) -> float:
    firsts = []
    for t in texts:
        words = normalize(t).split()
        skip = {"aaj", "main", "haan", "mein", "hai", "the", "a", "i", "is", "on", "and", "for"}
        for w in words:
            if w not in skip and len(w) >= 3:
                firsts.append(w[:3])
                break
    if len(firsts) < 2: return 0.0
    unique = len(set(firsts))
    return round(1.0 - (unique - 1) / len(firsts), 2)

def hallucination_risk(transcript: str, latency: float) -> float:
    t = normalize(transcript)
    word_count = len(t.split())
    localities_found = detect_localities(transcript)
    risk = 0.0
    filler_phrases = ["thank you for calling", "please hold", "how can i help", "this is", "welcome to", "i am calling"]
    for phrase in filler_phrases:
        if phrase in t: risk += 0.3
    if latency < 0.5 and word_count > 20: risk += 0.2
    if not localities_found and word_count > 3: risk += 0.2
    return round(min(risk, 1.0), 2)

def token_agreement_map(texts: list[str]) -> dict:
    if not texts: return {}
    word_sets = [set(normalize(t).split()) for t in texts if t.strip()]
    if not word_sets: return {}
    common = word_sets[0]
    for ws in word_sets[1:]:
        common &= ws
    return {w: sum(w in normalize(t).split() for t in texts) for w in common}

def transcribe_deepgram(wav_path: str) -> dict:
    best_result = {"text": "", "latency": 0, "confidence": None, "lang": "en-IN", "error": None}
    for lang in ["en-IN", "hi"]:
        url = (f"https://api.deepgram.com/v1/listen?model=nova-2&language={lang}&punctuate=true&utterances=true")
        headers = {"Authorization": f"Token {DEEPGRAM_API_KEY}", "Content-Type": "audio/wav"}
        t0 = time.time()
        try:
            with open(wav_path, "rb") as f:
                resp = requests.post(url, headers=headers, data=f, timeout=30)
            latency = round(time.time() - t0, 3)
            if resp.status_code != 200:
                best_result["error"] = f"HTTP {resp.status_code}: {resp.text[:80]}"
                continue
            data = resp.json()
            alt  = data["results"]["channels"][0]["alternatives"][0]
            text = alt.get("transcript", "")
            conf = alt.get("confidence")
            if text.strip():
                return {"text": text, "latency": latency, "confidence": conf, "lang": lang, "error": None}
        except Exception as e:
            best_result["error"] = str(e)
    return best_result

def load_whisper() -> WhisperModel:
    configs = [
        ("large-v3", "cuda", "float16"),
        ("medium",   "cuda", "float16"),
        ("medium",   "cpu",  "int8"),
        ("small",    "cpu",  "int8"),
    ]
    for model_size, device, dtype in configs:
        try:
            m = WhisperModel(model_size, device=device, compute_type=dtype)
            print(f"✓ Loaded Whisper {model_size} on {device} [{dtype}]")
            return m, model_size
        except Exception:
            continue
    raise RuntimeError("Could not load any Whisper model")

def transcribe_whisper(model: WhisperModel, wav_path: str) -> dict:
    t0 = time.time()
    try:
        segs, info = model.transcribe(
            wav_path, task="translate", beam_size=5, vad_filter=True,
            vad_parameters={"min_silence_duration_ms": 500}, temperature=0.0
        )
        text = " ".join(s.text for s in segs).strip()
        latency = round(time.time() - t0, 3)
        return {"text": text, "latency": latency, "lang": info.language, "lang_prob": round(info.language_probability, 3), "error": None}
    except Exception as e:
        return {"text": "", "latency": round(time.time() - t0, 3), "lang": None, "lang_prob": 0, "error": str(e)}

def transcribe_sarvam(wav_path: str) -> dict:
    url = "https://api.sarvam.ai/speech-to-text"
    headers = {"api-subscription-key": SARVAM_API_KEY}
    t0 = time.time()
    try:
        with open(wav_path, "rb") as f:
            resp = requests.post(
                url, headers=headers,
                files={"file": (os.path.basename(wav_path), f, "audio/wav")},
                data={"language_code": "hi-IN", "model": "saarika:v2.5", "with_timestamps": "false"},
                timeout=30,
            )
        latency = round(time.time() - t0, 3)
        if resp.status_code != 200:
            return {"text": "", "latency": latency, "error": f"HTTP {resp.status_code}: {resp.text[:80]}"}
        data = resp.json()
        return {"text": data.get("transcript", ""), "latency": latency, "error": None}
    except Exception as e:
        return {"text": "", "latency": round(time.time() - t0, 3), "error": str(e)}

In [16]:
def print_summary(df: pd.DataFrame, whisper_size: str = "medium"):
    MODELS = [(f"Deepgram nova-2", "dg"), (f"Whisper {whisper_size}", "wh"), ("Sarvam saarika:v2.5", "sv")]
    print("\n" + "═"*65)
    print("BENCHMARK SUMMARY  (No Ground Truth)")
    print("═"*65)

    print(f"\n{'Metric':<35} {'Deepgram':>10} {'Whisper':>10} {'Sarvam':>10}")
    print("─"*65)
    metrics = [
        ("Locality Detection Rate",   "locality_detected", "%"),
        ("Avg Latency (s)",           "latency_s",        "s"),
        ("Avg Word Count",            "word_count",       ""),
        ("Avg Hallucination Risk",    "hallucination_risk",""),
    ]
    for label, key, unit in metrics:
        vals = []
        for _, p in MODELS:
            col = f"{p}_{key}"
            v = df[col].mean() if col in df else float("nan")
            if unit == "%" : vals.append(f"{v*100:.1f}%")
            elif unit == "s": vals.append(f"{v:.2f}s")
            else: vals.append(f"{v:.2f}")
        print(f"  {label:<33} {vals[0]:>10} {vals[1]:>10} {vals[2]:>10}")

    print(f"\n  Cross-Model Agreement (avg)  : {df['cross_model_agreement'].mean():.1f}/100")
    print(f"  Transcript Entropy (avg)     : {df['transcript_entropy'].mean():.3f}  (lower = models agree)")
    print(f"  Phoneme Stability Score(avg) : {df['phoneme_stability_score'].mean():.2f}  (higher = stable start)")

    print("\n── Latency Summary ──")
    for name, p in MODELS:
        col = f"{p}_latency_s"
        if col in df:
            vals = df[col].dropna()
            print(f"  {name:<26}  P50={vals.quantile(0.5):.2f}s  P95={vals.quantile(0.95):.2f}s  Max={vals.max():.2f}s")
    print("═"*65)

def generate_html_report(df: pd.DataFrame, whisper_size: str = "medium"):
    n = len(df)
    def pct(col): return f"{df[col].mean()*100:.1f}%" if col in df else "N/A"
    def avg(col, fmt=".2f"): return f"{df[col].mean():{fmt}}" if col in df else "N/A"

    file_rows = ""
    for _, row in df.iterrows():
        cmar_color = "#06d6a0" if row["cross_model_agreement"] >= 70 else "#f77f00" if row["cross_model_agreement"] >= 40 else "#e63946"
        file_rows += f"""
        <tr>
          <td><strong>{row['file']}</strong><br><small class="tag">{row['noise_condition']}</small></td>
          <td style="color:{cmar_color};font-weight:700">{row['cross_model_agreement']:.0f}</td>
          <td>{row['transcript_entropy']:.3f}</td>
          <td>{row['consensus_localities'] or '—'}</td>
          <td class="{'hit' if row['dg_locality_detected'] else 'miss'}">{row['dg_localities'] or '✗'}</td>
          <td class="{'hit' if row['wh_locality_detected'] else 'miss'}">{row['wh_localities'] or '✗'}</td>
          <td class="{'hit' if row['sv_locality_detected'] else 'miss'}">{row['sv_localities'] or '✗'}</td>
        </tr>
        <tr class="transcript-row">
          <td colspan="7">
            <div class="transcript-grid">
              <div><span class="model-tag dg">Deepgram</span> {row['dg_transcript'] or '<em>empty</em>'}</div>
              <div><span class="model-tag wh">Whisper</span> {row['wh_transcript'] or '<em>empty</em>'}</div>
              <div><span class="model-tag sv">Sarvam</span> {row['sv_transcript'] or '<em>empty</em>'}</div>
            </div>
          </td>
        </tr>
        """

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<style>
  @import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;600&family=IBM+Plex+Sans:wght@300;400;600;700&display=swap');
  :root {{ --bg: #0d1117; --surface: #161b22; --border: #30363d; --text: #e6edf3; --muted: #8b949e; --dg: #00b4d8; --wh: #f77f00; --sv: #06d6a0; --red: #e63946; --green: #06d6a0; }}
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ background: var(--bg); color: var(--text); font-family: 'IBM Plex Sans', sans-serif; font-size: 14px; line-height: 1.6; padding: 40px 24px; }}
  .container {{ max-width: 1100px; margin: 0 auto; }}
  h1 {{ font-size: 2rem; font-weight: 700; margin-bottom: 4px; }}
  h2 {{ font-size: 1.1rem; color: var(--muted); margin: 32px 0 16px; text-transform: uppercase; border-bottom: 1px solid var(--border); padding-bottom: 8px; }}
  .cards {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(200px, 1fr)); gap: 16px; margin-bottom: 32px; }}
  .card {{ background: var(--surface); border: 1px solid var(--border); border-radius: 8px; padding: 20px; }}
  .card .val {{ font-size: 2rem; font-weight: 700; font-family: 'IBM Plex Mono', monospace; }}
  .card .label {{ color: var(--muted); font-size: 0.75rem; text-transform: uppercase; margin-top: 4px; }}
  table {{ width: 100%; border-collapse: collapse; background: var(--surface); border-radius: 8px; overflow: hidden; }}
  th {{ background: #1f2428; color: var(--muted); font-size: 0.72rem; text-transform: uppercase; padding: 10px 14px; text-align: left; }}
  td {{ padding: 10px 14px; border-bottom: 1px solid var(--border); vertical-align: top; }}
  .hit {{ color: var(--green); font-weight: 600; }}
  .miss {{ color: var(--red); }}
  .model-tag {{ display: inline-block; padding: 1px 7px; border-radius: 10px; font-size: 0.7rem; font-weight: 600; margin-right: 6px; }}
  .model-tag.dg {{ background: rgba(0,180,216,0.15); color: var(--dg); }}
  .model-tag.wh {{ background: rgba(247,127,0,0.15); color: var(--wh); }}
  .model-tag.sv {{ background: rgba(6,214,160,0.15); color: var(--sv); }}
  .transcript-row td {{ background: #0d1117; padding: 8px 14px; }}
  .transcript-grid {{ display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 12px; font-size: 0.82rem; color: var(--muted); }}
</style>
</head>
<body>
<div class="container">
<h1>ASR Benchmark Report</h1>
<p style="color: var(--muted); margin-bottom: 40px; font-size: 0.9rem;">Indian Conversational Speech · Bangalore Localities · {n} audio samples</p>

<h2>Model Summary</h2>
<div class="cards">
  <div class="card" style="border-top: 3px solid var(--dg);"><div class="val">{pct('dg_locality_detected')}</div><div class="label">Deepgram LDR</div></div>
  <div class="card" style="border-top: 3px solid var(--wh);"><div class="val">{pct('wh_locality_detected')}</div><div class="label">Whisper LDR</div></div>
  <div class="card" style="border-top: 3px solid var(--sv);"><div class="val">{pct('sv_locality_detected')}</div><div class="label">Sarvam LDR</div></div>
  <div class="card"><div class="val">{avg('cross_model_agreement', '.0f')}</div><div class="label">Model Agreement /100</div></div>
</div>

<h2>Per-File Results</h2>
<table>
  <thead>
    <tr>
      <th>File</th><th>Agreement</th><th>Entropy</th><th>Consensus</th><th>Deepgram</th><th>Whisper</th><th>Sarvam</th>
    </tr>
  </thead>
  <tbody>
    {file_rows}
  </tbody>
</table>
</div>
</body>
</html>"""

    with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
        f.write(html)
    print(f" Saved HTML report → {OUTPUT_HTML}")


def run():
    print("="*60)
    print("ASR BENCHMARK")
    print("="*60)
    print("\nLoading Whisper model...")
    whisper_model, whisper_size = load_whisper()

    wav_files = sorted([f for f in os.listdir(AUDIO_DIR) if f.lower().endswith(".wav")])
    if not wav_files:
        print(f"\n No .wav files found in {AUDIO_DIR}. Please upload files and run again.")
        return pd.DataFrame()

    print(f"\nFound {len(wav_files)} WAV files")
    print("─"*60)

    rows = []
    for filename in wav_files:
        stem = Path(filename).stem
        path = os.path.join(AUDIO_DIR, filename)
        noise = NOISE_CONDITIONS.get(stem, DEFAULT_NOISE)
        print(f"\n▶  {filename}  [{noise}]")

        dg = transcribe_deepgram(path)
        wh = transcribe_whisper(whisper_model, path)
        sv = transcribe_sarvam(path)

        # Convert Hindi/Devanagari text to Roman alphabet for metric comparisons
        sv_text_roman = unidecode(sv["text"]) if sv["text"] else ""

        # Calculate cross-model metrics using the Romanized version of Sarvam
        metric_texts = [t for t in [dg["text"], wh["text"], sv_text_roman] if t.strip()]

        cmar  = cross_model_agreement(metric_texts)
        entr  = transcript_entropy(metric_texts)
        pss   = phoneme_stability_score(metric_texts)
        common_tokens = token_agreement_map(metric_texts)

        # But use the ORIGINAL Hindi text for locality detection (it will hit our new Hindi dictionary keys)
        dg_locs = detect_localities(dg["text"])
        wh_locs = detect_localities(wh["text"])
        sv_locs = detect_localities(sv["text"])
        if stem in MANUAL_LOCALITIES:
            consensus_locs = MANUAL_LOCALITIES[stem]
            print(f"    Manual GT Localities → {consensus_locs}")
        else:
            # Fallback to auto-majority vote if no manual entry exists
            consensus_locs = sorted(set(dg_locs) & set(wh_locs) | set(dg_locs) & set(sv_locs) | set(wh_locs) & set(sv_locs))
            print(f"    Auto-Consensus → {consensus_locs}")

        rows.append({
            "file": filename, "stem": stem, "noise_condition": noise,
            "cross_model_agreement": cmar, "transcript_entropy": entr, "phoneme_stability_score": pss,
            "consensus_localities": "|".join(consensus_locs),
            "dg_transcript": dg["text"], "dg_latency_s": dg["latency"], "dg_localities": "|".join(dg_locs), "dg_locality_detected": 1 if dg_locs else 0, "dg_hallucination_risk": hallucination_risk(dg["text"], dg["latency"]),
            "wh_transcript": wh["text"], "wh_latency_s": wh["latency"], "wh_localities": "|".join(wh_locs), "wh_locality_detected": 1 if wh_locs else 0, "wh_hallucination_risk": hallucination_risk(wh["text"], wh["latency"]),
            "sv_transcript": sv["text"], "sv_latency_s": sv["latency"], "sv_localities": "|".join(sv_locs), "sv_locality_detected": 1 if sv_locs else 0, "sv_hallucination_risk": hallucination_risk(sv["text"], sv["latency"]),
        })

    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_CSV, index=False)
    print_summary(df, whisper_size)
    generate_html_report(df, whisper_size)
    return df

In [17]:
# ── CELL 5: EXECUTION & VISUALIZATION ──
import IPython

df_results = run()

if not df_results.empty:
    display(IPython.display.HTML(filename=OUTPUT_HTML))

ASR BENCHMARK

Loading Whisper model...
✓ Loaded Whisper large-v3 on cuda [float16]

Found 20 WAV files
────────────────────────────────────────────────────────────

▶  Bellandur.wav  [quiet]
    Manual GT Localities → ['Bellandur']

▶  Btm layout.wav  [quiet]
    Manual GT Localities → ['BTM Layout']

▶  Church street.wav  [quiet]
    Auto-Consensus → []

▶  Electronic city.wav  [noisy]
    Manual GT Localities → ['Electronic city']

▶  Electronic city2.wav  [quite]
    Manual GT Localities → ['Electronic city']

▶  Hsr.wav  [nosiy]
    Manual GT Localities → ['HSR Layout']

▶  Jp nagar.wav  [quiet]
    Manual GT Localities → ['JP Nagar']

▶  Kengeri.wav  [quiet]
    Manual GT Localities → ['Kengeri']

▶  Koramangala.wav  [quiet]
    Manual GT Localities → ['Koramangala']

▶  MTR hotel.wav  [nosiy]
    Manual GT Localities → ['MTR Hotel']

▶  Majestic,silk board, koramangala.wav  [quiet]
    Manual GT Localities → ['Majestic', 'Silk Board', 'Koramangala']

▶  Majestic.wav  [quiet]
   